# Notebook Overview — Prepare Video Evidence

## Purpose

This notebook prepares lightweight video evidence metadata for the NExT-QA VideoQA representation-learning project. Instead of duplicating video content, the notebook generates evidence records that identify meaningful regions of the original videos using video identifiers, timestamps, segment boundaries, and related metadata.

The notebook is independently runnable in a fresh Colab runtime. During initialization, it verifies that the NExT-QA videos are available locally. If the local video cache is missing, the notebook restores it from the preferred Google Drive release artifact, `releases/NExTVideo_combined.zip`. If the combined archive is unavailable, the notebook can fall back to the legacy multipart archive workflow.

The resulting evidence metadata serves as the bridge between the raw NExT-QA video dataset and the representation-learning workflows introduced in subsequent notebooks. These evidence records provide the structured video segments used for latent feature extraction, pretrained representation generation, and downstream VideoQA experimentation.

## Inputs

* Preferred NExT-QA combined video archive stored in Google Drive

  * `releases/NExTVideo_combined.zip`

* Legacy NExT-QA multipart video archive files stored in Google Drive, used only as fallback

  * `releases/NExTVideo.z01`
  * `releases/NExTVideo.z02`
  * `releases/NExTVideo.z03`
  * `releases/NExTVideo.z04`
  * `releases/NExTVideo.z05`
  * `releases/NExTVideo.z06`
  * `releases/NExTVideo.zip`

* NExT-QA question-answer files

  * `train.csv`
  * `val.csv`
  * `test.csv`

* NExT-QA metadata resources
* Project configuration settings
* Shared video, metadata, evidence, and validation utility functions

## Outputs

* Restored local NExT-QA video cache
* Evidence metadata CSV file
* Video inventory summary
* Evidence metadata summary report
* Evidence validation report
* Sample evidence records for verification

## Processing Workflow

1. Load project configuration and dataset resources.
2. Restore or verify the local NExT-QA video cache.
3. Load NExT-QA metadata and video inventory information.
4. Define the evidence metadata schema and segmentation parameters.
5. Inspect representative videos and extract video properties.
6. Generate evidence metadata records for each processed video.
7. Validate evidence metadata completeness and consistency.
8. Save evidence metadata and summary files.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True

import os
from google.colab import userdata

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository directories...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --cone

    !git sparse-checkout set \
        src \
        datasets \
        outputs

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "datasets",
    "outputs",
    "datasets/NExT-QA",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
    "src/videoqa_representation_config.py",
    "src/nextqa_video_cache.py",
    "src/nextqa_metadata.py",
    "src/video_evidence.py",
    "src/evidence_validation.py",
    "src/evidence_io.py",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nRepository directories:")
    !find src datasets outputs \
        -maxdepth 2 \
        -type d \
        ! -path "*/__pycache__*" | sort



### 🔷 Step 2 — Import Libraries and Load Configuration

* Import the Python libraries required for evidence metadata generation and validation.
* Load centralized project configuration settings and constants from `videoqa_representation_config.py`.
* Import reusable video utility functions from `nextqa_video_cache.py`.
* Initialize shared configuration values, paths, and runtime settings used throughout the notebook.
* Verify that required modules and configuration resources are available before continuing.


In [ ]:
# ============================================================
# Step 2: Import Libraries and Load Configuration
# ============================================================

# ------------------------------------------------------------
# Standard Library Imports
# ------------------------------------------------------------

from pathlib import Path

# ------------------------------------------------------------
# Third-Party Library Imports
# ------------------------------------------------------------

import pandas as pd

# ------------------------------------------------------------
# Project Configuration
# ------------------------------------------------------------

from src.videoqa_representation_config import *

# ------------------------------------------------------------
# Reusable Project Modules
# ------------------------------------------------------------

from src.nextqa_video_cache import *
from src.nextqa_metadata import *
from src.video_evidence import *
from src.evidence_validation import *
from src.evidence_io import *

# ------------------------------------------------------------
# Import Verification
# ------------------------------------------------------------

print("Project configuration loaded successfully.")
print("Project utility modules loaded successfully.")

if VERBOSE:
    print("\nLoaded Modules:")
    print("  ✓ iterative_rag_config")
    print("  ✓ nextqa_video_cache")
    print("  ✓ nextqa_metadata")
    print("  ✓ video_evidence")
    print("  ✓ evidence_validation")
    print("  ✓ evidence_io")



### 🔷 Step 3 — Define Input and Output Paths

* Define the input directories containing NExT-QA videos, questions, and metadata resources.
* Define the output directories used to store evidence metadata and validation reports.
* Construct notebook paths using centralized project configuration values.
* Create required output directories when they do not already exist.
* Verify that required input paths are available before continuing.


In [ ]:
# ============================================================
# Step 3: Define Input and Output Paths
# ============================================================

# ------------------------------------------------------------
# NExT-QA Input Directories
# ------------------------------------------------------------

INPUT_QUESTIONS_DIR = QUESTIONS_DIR
INPUT_METADATA_DIR = METADATA_DIR
INPUT_VIDEOS_DIR = VIDEOS_DIR

# ------------------------------------------------------------
# Evidence Output Directories
# ------------------------------------------------------------

EVIDENCE_METADATA_DIR = EVIDENCE_DIR / "metadata"
EVIDENCE_REPORTS_DIR = EVIDENCE_DIR / "reports"

EVIDENCE_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Evidence Output Files
# ------------------------------------------------------------

EVIDENCE_METADATA_CSV = (
    EVIDENCE_METADATA_DIR /
    "evidence_metadata.csv"
)

EVIDENCE_VALIDATION_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_validation.csv"
)

EVIDENCE_SUMMARY_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_summary.csv"
)

# ------------------------------------------------------------
# Verify Required Input Paths
# ------------------------------------------------------------

required_input_paths = [
    INPUT_QUESTIONS_DIR,
    INPUT_METADATA_DIR,
]

for path in required_input_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required input path not found: {path}"
        )

print("Input and output paths initialized successfully.")

if VERBOSE:
    print("\nInput Directories")
    print("-" * 60)
    print(f"Questions : {INPUT_QUESTIONS_DIR}")
    print(f"Metadata  : {INPUT_METADATA_DIR}")
    print(f"Videos    : {INPUT_VIDEOS_DIR}")

    print("\nOutput Directories")
    print("-" * 60)
    print(f"Evidence Metadata : {EVIDENCE_METADATA_DIR}")
    print(f"Evidence Reports  : {EVIDENCE_REPORTS_DIR}")

    print("\nOutput Files")
    print("-" * 60)
    print(f"Evidence CSV : {EVIDENCE_METADATA_CSV}")
    print(f"Summary CSV  : {EVIDENCE_SUMMARY_CSV}")



### 🔷 Step 4 — Restore Local NExT-QA Video Cache

* Verify whether the NExT-QA video cache is already available in local Colab storage.
* Mount Google Drive and locate the NExT-QA release resources when local videos are missing.
* Prefer the combined NExT-QA release archive when available:
  * `releases/NExTVideo_combined.zip`
* Copy the combined archive to the local dataset archive workspace only when needed.
* Validate the local combined archive before extraction.
* Fall back to the legacy multipart archive workflow only when the combined archive is unavailable.
* Extract the NExT-QA video archive into local Colab storage when the video cache is missing.
* Verify that the restored video cache contains the expected NExT-QA video files and folder structure.
* Prepare the local video dataset for evidence generation in subsequent steps.


In [ ]:
# ============================================================
# Step 4: Restore Local NExT-QA Video Cache
# ============================================================

import shutil
import time

from google.colab import drive

# ------------------------------------------------------------
# Expected Video Cache Size
# ------------------------------------------------------------

EXPECTED_NEXTQA_VIDEO_COUNT = 5440

# ------------------------------------------------------------
# Check Existing Local Video Cache
# ------------------------------------------------------------

existing_video_files = sorted(
    INPUT_VIDEOS_DIR.rglob("*.mp4")
)

if len(existing_video_files) == EXPECTED_NEXTQA_VIDEO_COUNT:

    video_cache_restore_summary = {
        "cache_status": "already_available",
        "video_count": len(existing_video_files),
        "local_videos_dir": str(INPUT_VIDEOS_DIR),
        "archive_mode": "not_required",
        "verified": True,
    }

    print("Local NExT-QA video cache already available.")
    print(f"Video files found: {len(existing_video_files)}")
    print(f"Local video directory: {INPUT_VIDEOS_DIR}")

else:

    print("Local NExT-QA video cache is missing or incomplete.")
    print(f"Video files found locally: {len(existing_video_files)}")
    print("Restoring video cache from Google Drive release archive...")

    # --------------------------------------------------------
    # Mount Google Drive
    # --------------------------------------------------------

    GOOGLE_DRIVE_MOUNT = "/content/drive"

    if not os.path.exists(GOOGLE_DRIVE_MOUNT):

        if VERBOSE:
            print("\nMounting Google Drive...")

        drive.mount(GOOGLE_DRIVE_MOUNT)

    else:

        if VERBOSE:
            print("\nGoogle Drive is already mounted.")

    drive_root = Path(GOOGLE_DRIVE_MOUNT) / "MyDrive"

    if not drive_root.exists():

        raise FileNotFoundError(
            "Unable to access Google Drive root directory."
        )

    # --------------------------------------------------------
    # Configure Google Drive Release and Local Cache Paths
    # --------------------------------------------------------

    DRIVE_DATASET_DIR = drive_root / "VideoQA_Project" / "NExT-QA"
    DRIVE_RELEASES_DIR = DRIVE_DATASET_DIR / "releases"

    LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"
    LOCAL_ARCHIVE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    COMBINED_ARCHIVE_NAME = "NExTVideo_combined.zip"

    DRIVE_COMBINED_ARCHIVE_PATH = (
        DRIVE_RELEASES_DIR /
        COMBINED_ARCHIVE_NAME
    )

    COMBINED_ARCHIVE_PATH = (
        LOCAL_ARCHIVE_DIR /
        COMBINED_ARCHIVE_NAME
    )

    required_archive_files = [
        "NExTVideo.z01",
        "NExTVideo.z02",
        "NExTVideo.z03",
        "NExTVideo.z04",
        "NExTVideo.z05",
        "NExTVideo.z06",
        "NExTVideo.zip",
    ]

    if not DRIVE_RELEASES_DIR.exists():

        raise FileNotFoundError(
            "Google Drive NExT-QA releases directory not found:\n"
            f"{DRIVE_RELEASES_DIR}"
        )

    # --------------------------------------------------------
    # Prefer Combined Archive
    # --------------------------------------------------------

    if DRIVE_COMBINED_ARCHIVE_PATH.exists():

        print("\nPreferred combined NExT-QA archive found.")
        print(f"Source archive : {DRIVE_COMBINED_ARCHIVE_PATH}")
        print(f"Local archive  : {COMBINED_ARCHIVE_PATH}")

        source_size = DRIVE_COMBINED_ARCHIVE_PATH.stat().st_size

        copy_start_time = time.time()

        if COMBINED_ARCHIVE_PATH.exists():

            local_size = COMBINED_ARCHIVE_PATH.stat().st_size

            if local_size == source_size:

                print("Local combined archive already exists with matching size.")
                print("Archive copy skipped.")

            else:

                print(
                    "Local combined archive exists but size differs. "
                    "Replacing local archive."
                )

                COMBINED_ARCHIVE_PATH.unlink()

                shutil.copy2(
                    DRIVE_COMBINED_ARCHIVE_PATH,
                    COMBINED_ARCHIVE_PATH,
                )

        else:

            print("Copying preferred combined archive to local storage...")

            shutil.copy2(
                DRIVE_COMBINED_ARCHIVE_PATH,
                COMBINED_ARCHIVE_PATH,
            )

        copy_elapsed_time = time.time() - copy_start_time

        local_size = COMBINED_ARCHIVE_PATH.stat().st_size

        if local_size != source_size:

            raise ValueError(
                "Combined archive copy failed size verification."
            )

        archive_restore_summary = {
            "archive_mode": "combined",
            "source_archive": str(DRIVE_COMBINED_ARCHIVE_PATH),
            "local_archive": str(COMBINED_ARCHIVE_PATH),
            "archive_size_gb": local_size / (1024 ** 3),
            "copy_elapsed_seconds": copy_elapsed_time,
            "verified": True,
        }

        print("Combined archive copied and verified.")
        print(f"Elapsed time : {copy_elapsed_time:.1f} seconds")
        print(f"Local size   : {local_size / (1024 ** 3):.2f} GB")

    # --------------------------------------------------------
    # Fallback: Legacy Multipart Archive Workflow
    # --------------------------------------------------------

    else:

        print(
            "\nPreferred combined archive not found. "
            "Falling back to legacy multipart archive workflow."
        )

        archive_verification_summary = verify_nextqa_archive_parts(
            archive_parts_dir=DRIVE_RELEASES_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        local_archive_summary = copy_nextqa_archive_parts_to_local(
            source_archive_dir=DRIVE_RELEASES_DIR,
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            required_archive_files=required_archive_files,
            verbose=VERBOSE,
        )

        archive_restore_summary = build_combined_nextqa_archive(
            local_archive_dir=LOCAL_ARCHIVE_DIR,
            combined_archive_path=COMBINED_ARCHIVE_PATH,
            split_archive_name="NExTVideo.zip",
            required_archive_files=required_archive_files,
            force_rebuild=True,
            verbose=VERBOSE,
        )

    # --------------------------------------------------------
    # Extract Combined Archive and Verify Video Cache
    # --------------------------------------------------------

    print("\nExtracting or verifying local NExT-QA video cache...")

    extraction_start_time = time.time()

    extract_summary = extract_nextqa_video_archive(
        combined_archive_path=COMBINED_ARCHIVE_PATH,
        local_videos_dir=INPUT_VIDEOS_DIR,
        force_extract=False,
        verbose=VERBOSE,
    )

    extraction_elapsed_time = time.time() - extraction_start_time

    restored_video_files = sorted(
        INPUT_VIDEOS_DIR.rglob("*.mp4")
    )

    if len(restored_video_files) != EXPECTED_NEXTQA_VIDEO_COUNT:

        raise ValueError(
            "NExT-QA video cache verification failed. "
            f"Expected {EXPECTED_NEXTQA_VIDEO_COUNT} videos, "
            f"found {len(restored_video_files)}."
        )

    video_cache_restore_summary = {
        "cache_status": "restored",
        "video_count": len(restored_video_files),
        "local_videos_dir": str(INPUT_VIDEOS_DIR),
        "archive_mode": archive_restore_summary.get(
            "archive_mode",
            "unknown",
        ),
        "archive_restore_summary": archive_restore_summary,
        "extract_summary": extract_summary,
        "extraction_elapsed_seconds": extraction_elapsed_time,
        "verified": True,
    }

    print("\nLocal NExT-QA video cache restored and verified.")
    print(f"Video files found : {len(restored_video_files)}")
    print(f"Elapsed time      : {extraction_elapsed_time:.1f} seconds")
    print(f"Local videos      : {INPUT_VIDEOS_DIR}")

print("\nLocal NExT-QA video cache is ready.")


### 🔷 Step 5 — Load NExT-QA Metadata and Video Inventory

* Load NExT-QA question-answer files and supporting dataset metadata.
* Load the video inventory and identify videos available for processing.
* Associate video identifiers with dataset splits and metadata records.
* Verify that required metadata resources contain valid records.
* Generate summary statistics for videos and question-answer datasets.



In [ ]:
# ============================================================
# Step 5: Load NExT-QA Metadata and Video Inventory
# ============================================================

# ------------------------------------------------------------
# Load NExT-QA Annotation Files
# ------------------------------------------------------------

split_annotations = load_nextqa_split_annotations(
    annotations_dir=INPUT_QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Build Local Video Inventory
# ------------------------------------------------------------

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=INPUT_VIDEOS_DIR,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Attach Video Inventory Information
# ------------------------------------------------------------

annotations_with_videos_df = (
    attach_video_inventory_to_annotations(
        annotations=annotations_df,
        video_inventory=video_inventory_df,
        verbose=VERBOSE,
    )
)

# ------------------------------------------------------------
# Generate Split Summary
# ------------------------------------------------------------

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

# ------------------------------------------------------------
# Verify Annotation Coverage
# ------------------------------------------------------------

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nNExT-QA metadata and video inventory loaded successfully.")

print(f"Annotation records : {len(annotations_df):,}")
print(f"Video inventory    : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)



### 🔷 Step 6 — Define Evidence Metadata Schema

* Define the structure and fields used to represent video evidence records.
* Specify required metadata attributes, identifiers, timestamps, and evidence relationships.
* Establish data types and validation requirements for evidence records.
* Ensure the schema supports downstream representation learning, analysis, and VideoQA workflows.
* Create the evidence metadata template used throughout the notebook.



In [ ]:
# ============================================================
# Step 6: Define Evidence Metadata Schema
# ============================================================

# ------------------------------------------------------------
# Required Evidence Metadata Columns
# ------------------------------------------------------------

EVIDENCE_SCHEMA = {
    "evidence_id": "str",
    "video_id": "str",
    "split": "str",
    "video_path": "str",
    "segment_index": "int",

    "evidence_level": "int",
    "parent_evidence_id": "str",
    "segment_strategy": "str",

    "start_time_sec": "float",
    "midpoint_time_sec": "float",
    "end_time_sec": "float",

    "segment_duration_sec": "float",

    "start_frame_idx": "int",
    "midpoint_frame_idx": "int",
    "end_frame_idx": "int",

    "representative_frame_index": "int",

    "fps": "float",
    "frame_count": "int",
    "width": "int",
    "height": "int",

    "motion_score": "float",
    "scene_change_score": "float",

    "created_by_notebook": "str",
}

EVIDENCE_COLUMNS = list(EVIDENCE_SCHEMA.keys())

# ------------------------------------------------------------
# Required Columns for Validation
# ------------------------------------------------------------

REQUIRED_EVIDENCE_COLUMNS = [
    "evidence_id",
    "video_id",
    "split",
    "video_path",

    "start_time_sec",
    "midpoint_time_sec",
    "end_time_sec",

    "segment_duration_sec",

    "start_frame_idx",
    "midpoint_frame_idx",
    "end_frame_idx",

    "representative_frame_index",
]

# ------------------------------------------------------------
# Columns Expected to Contain Unique Values
# ------------------------------------------------------------

UNIQUE_EVIDENCE_COLUMNS = [
    "evidence_id",
]

# ------------------------------------------------------------
# Columns Used for Downstream Retrieval
# ------------------------------------------------------------

RETRIEVAL_REFERENCE_COLUMNS = [
    "evidence_id",
    "video_id",
    "video_path",

    "start_time_sec",
    "midpoint_time_sec",
    "end_time_sec",

    "start_frame_idx",
    "midpoint_frame_idx",
    "end_frame_idx",

    "representative_frame_index",

    "motion_score",
]

# ------------------------------------------------------------
# Display Schema Summary
# ------------------------------------------------------------

print("Evidence metadata schema defined successfully.")
print(f"Schema columns: {len(EVIDENCE_COLUMNS)}")
print(f"Required columns: {len(REQUIRED_EVIDENCE_COLUMNS)}")
print(f"Unique columns: {len(UNIQUE_EVIDENCE_COLUMNS)}")

if VERBOSE:

    print("\nEvidence Metadata Columns")
    print("-" * 60)

    for column_name, data_type in EVIDENCE_SCHEMA.items():
        print(f"{column_name:<32} {data_type}")



#### Evidence Metadata Field Definitions

| Field | Description |
|---------|-------------|
| `evidence_id` | Unique identifier assigned to each evidence unit. |
| `video_id` | NExT-QA video identifier associated with the evidence unit. |
| `split` | Dataset split associated with the source video or related QA records (`train`, `val`, or `test`). |
| `video_path` | Local path to the source video file used to generate the evidence unit. |
| `segment_index` | Sequential index of the evidence segment within the source video. |
| `evidence_level` | Hierarchy level of the evidence unit. Level `0` typically represents a parent or top-level segment. |
| `parent_evidence_id` | Identifier of the parent evidence unit when hierarchical segmentation is used. Empty for top-level evidence units. |
| `segment_strategy` | Segmentation method used to create the evidence unit, such as fixed-duration, scene-based, or motion-based segmentation. |
| `start_time_sec` | Segment start time in seconds from the beginning of the source video. |
| `midpoint_time_sec` | Segment midpoint time in seconds, used as a representative temporal reference. |
| `end_time_sec` | Segment end time in seconds from the beginning of the source video. |
| `segment_duration_sec` | Duration of the evidence segment in seconds. |
| `start_frame_idx` | Frame index corresponding to the segment start time. |
| `midpoint_frame_idx` | Frame index corresponding to the segment midpoint time. |
| `end_frame_idx` | Frame index corresponding to the segment end time. |
| `representative_frame_index` | Frame index selected as the representative visual frame for the evidence segment. The midpoint frame is currently used for fixed-duration segmentation. |
| `fps` | Frames per second of the source video. |
| `frame_count` | Total number of frames in the source video. |
| `width` | Source video frame width in pixels. |
| `height` | Source video frame height in pixels. |
| `motion_score` | Normalized estimate of motion or visual activity within the evidence segment. A value of 0.0 indicates motion scoring was not computed or no measurable motion was detected. |
| `scene_change_score` | Numeric estimate of scene-transition strength associated with the evidence segment. Currently populated with a default placeholder value for future scene-analysis workflows. |
| `created_by_notebook` | Notebook identifier used to record which notebook generated the evidence metadata. |

### 🔷 Step 7 — Define Evidence Segmentation Parameters

* Define the parameters used to partition videos into evidence segments.
* Specify segment duration limits, sampling intervals, and boundary selection criteria.
* Configure start, midpoint, and end timestamp generation for each segment.
* Define parent-child relationships for hierarchical evidence segmentation.
* Establish segmentation settings used throughout evidence generation.



In [ ]:
# ============================================================
# Step 7: Define Evidence Segmentation Parameters
# ============================================================

# ------------------------------------------------------------
# Segmentation Strategy
# ------------------------------------------------------------
SEGMENT_STRATEGY = "fixed_duration"

# ------------------------------------------------------------
# Segment Duration Settings
# ------------------------------------------------------------
MIN_SEGMENT_DURATION_SEC = 4.0
MAX_SEGMENT_DURATION_SEC = 8.0
DEFAULT_SEGMENT_DURATION_SEC = 6.0

# ------------------------------------------------------------
# Frame Reference Settings
# ------------------------------------------------------------
INCLUDE_START_FRAME = True
INCLUDE_MIDPOINT_FRAME = True
INCLUDE_END_FRAME = True

# ------------------------------------------------------------
# Evidence Hierarchy Settings
# ------------------------------------------------------------
ENABLE_PARENT_EVIDENCE = False
PARENT_SEGMENT_DURATION_SEC = None
EVIDENCE_LEVEL = 0

# ------------------------------------------------------------
# Motion and Scene Metrics
# ------------------------------------------------------------
COMPUTE_MOTION_SCORE = True
COMPUTE_SCENE_CHANGE_SCORE = False
DEFAULT_SCENE_CHANGE_SCORE = 0.0

# ------------------------------------------------------------
# Processing Limits
# ------------------------------------------------------------

# Use an integer for testing (e.g., 25, 50, 100)
# Use "ALL" to process the complete dataset

MAX_VIDEOS_TO_PROCESS = "ALL"
SAMPLE_VIDEO_COUNT = 5

# ------------------------------------------------------------
# Validate Parameter Settings
# ------------------------------------------------------------

if MIN_SEGMENT_DURATION_SEC <= 0:
    raise ValueError(
        "MIN_SEGMENT_DURATION_SEC must be greater than zero."
    )

if MAX_SEGMENT_DURATION_SEC < MIN_SEGMENT_DURATION_SEC:
    raise ValueError(
        "MAX_SEGMENT_DURATION_SEC must be greater than or equal to "
        "MIN_SEGMENT_DURATION_SEC."
    )

if not (
    MIN_SEGMENT_DURATION_SEC
    <= DEFAULT_SEGMENT_DURATION_SEC
    <= MAX_SEGMENT_DURATION_SEC
):
    raise ValueError(
        "DEFAULT_SEGMENT_DURATION_SEC must be between "
        "MIN_SEGMENT_DURATION_SEC and MAX_SEGMENT_DURATION_SEC."
    )

if (
    MAX_VIDEOS_TO_PROCESS != "ALL"
    and (
        not isinstance(MAX_VIDEOS_TO_PROCESS, int)
        or MAX_VIDEOS_TO_PROCESS <= 0
    )
):
    raise ValueError(
        "MAX_VIDEOS_TO_PROCESS must be a positive integer or 'ALL'."
    )

if SAMPLE_VIDEO_COUNT <= 0:
    raise ValueError(
        "SAMPLE_VIDEO_COUNT must be greater than zero."
    )

# ------------------------------------------------------------
# Assemble Parameter Summary
# ------------------------------------------------------------

EVIDENCE_SEGMENTATION_PARAMETERS = {
    "segment_strategy": SEGMENT_STRATEGY,
    "min_segment_duration_sec": MIN_SEGMENT_DURATION_SEC,
    "max_segment_duration_sec": MAX_SEGMENT_DURATION_SEC,
    "default_segment_duration_sec": DEFAULT_SEGMENT_DURATION_SEC,
    "include_start_frame": INCLUDE_START_FRAME,
    "include_midpoint_frame": INCLUDE_MIDPOINT_FRAME,
    "include_end_frame": INCLUDE_END_FRAME,
    "enable_parent_evidence": ENABLE_PARENT_EVIDENCE,
    "parent_segment_duration_sec": PARENT_SEGMENT_DURATION_SEC,
    "evidence_level": EVIDENCE_LEVEL,
    "compute_motion_score": COMPUTE_MOTION_SCORE,
    "compute_scene_change_score": COMPUTE_SCENE_CHANGE_SCORE,
    "default_scene_change_score": DEFAULT_SCENE_CHANGE_SCORE,
    "max_videos_to_process": MAX_VIDEOS_TO_PROCESS,
    "sample_video_count": SAMPLE_VIDEO_COUNT,
}

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("Evidence segmentation parameters defined successfully.")

print(f"Segment strategy : {SEGMENT_STRATEGY}")
print(
    f"Default duration : "
    f"{DEFAULT_SEGMENT_DURATION_SEC:.1f} seconds"
)
print(
    f"Duration range   : "
    f"{MIN_SEGMENT_DURATION_SEC:.1f}–"
    f"{MAX_SEGMENT_DURATION_SEC:.1f} seconds"
)
print(f"Parent evidence  : {ENABLE_PARENT_EVIDENCE}")
print(f"Motion scoring   : {COMPUTE_MOTION_SCORE}")
print(f"Videos processed : {MAX_VIDEOS_TO_PROCESS}")

if VERBOSE:
    print("\nEvidence Segmentation Parameters")
    print("-" * 60)
    for (
        parameter_name,
        parameter_value,
    ) in EVIDENCE_SEGMENTATION_PARAMETERS.items():
        print(
            f"{parameter_name:<32} "
            f"{parameter_value}"
        )



### 🔷 Step 8 — Inspect Sample Videos

* Select representative videos from the NExT-QA dataset for inspection.
* Extract basic video properties including duration, frame count, frame rate, and resolution.
* Verify that video files can be successfully opened and processed.
* Review video characteristics relevant to evidence generation.
* Generate summary statistics for the inspected videos.



In [ ]:
# ============================================================
# Step 8: Inspect Sample Videos
# ============================================================

# ------------------------------------------------------------
# Third-Party Video Library Imports
# ------------------------------------------------------------

import cv2

# ------------------------------------------------------------
# Select Sample Videos
# ------------------------------------------------------------

sample_video_inventory_df = (
    video_inventory_df
    .sample(
        n=min(SAMPLE_VIDEO_COUNT, len(video_inventory_df)),
        random_state=42,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Inspect Video Properties
# ------------------------------------------------------------

sample_video_records = []

for _, row in sample_video_inventory_df.iterrows():

    video_path = Path(row["video_path"])

    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():

        sample_video_records.append(
            {
                "video_id": row["video_id"],
                "video_path": str(video_path),
                "readable": False,
                "fps": None,
                "frame_count": None,
                "duration_sec": None,
                "width": None,
                "height": None,
            }
        )

        continue

    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration_sec = (
        frame_count / fps
        if fps and fps > 0
        else None
    )

    capture.release()

    sample_video_records.append(
        {
            "video_id": row["video_id"],
            "video_path": str(video_path),
            "readable": True,
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": duration_sec,
            "width": width,
            "height": height,
        }
    )

sample_video_properties_df = pd.DataFrame(sample_video_records)

# ------------------------------------------------------------
# Display Inspection Results
# ------------------------------------------------------------

readable_count = sample_video_properties_df["readable"].sum()

print("Sample video inspection complete.")
print(f"Sample videos inspected : {len(sample_video_properties_df)}")
print(f"Readable videos         : {readable_count}")

if VERBOSE:

    print("\nSample Video Properties")
    print("-" * 60)

    display(sample_video_properties_df)



### 🔷 Step 9 — Generate Evidence Metadata Records

* Process NExT-QA videos using the defined evidence segmentation parameters.
* Generate evidence records containing video identifiers, timestamps, and segment metadata.
* Assign unique identifiers and maintain links to source videos.
* Compute evidence attributes required for validation and downstream processing.
* Assemble evidence records into a structured dataset.


In [ ]:
COMPUTE_MOTION_SCORE = False

In [ ]:
# ============================================================
# Step 9: Generate Evidence Metadata Records
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Helper: Determine Split Assignment by Video
# ------------------------------------------------------------

video_split_lookup = (
    annotations_df
    .groupby("video_id")["split"]
    .apply(lambda values: ",".join(sorted(set(values.dropna()))))
    .to_dict()
)

# ------------------------------------------------------------
# Helper: Compute Motion Scores for All Segments in One Video
# ------------------------------------------------------------

def compute_video_segment_motion_scores(
    video_path,
    segment_frame_boundaries,
    max_sampled_frames_per_segment=12,
):
    """
    Compute lightweight normalized motion scores for all segments
    in a single video.

    The video is opened once. Each segment receives one motion score
    based on mean grayscale frame differences between sampled frames.

    Args:
        video_path:
            Path to the source video file.

        segment_frame_boundaries:
            List of tuples:
            (segment_index, start_frame_idx, end_frame_idx)

        max_sampled_frames_per_segment:
            Maximum number of frames sampled within each segment.

    Returns:
        dict: segment_index -> normalized motion score in range 0.0 to 1.0
    """

    motion_scores = {}

    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():

        for segment_index, _, _ in segment_frame_boundaries:
            motion_scores[segment_index] = 0.0

        return motion_scores

    for (
        segment_index,
        start_frame_idx,
        end_frame_idx,
    ) in segment_frame_boundaries:

        if end_frame_idx <= start_frame_idx:

            motion_scores[segment_index] = 0.0
            continue

        sampled_frame_indices = np.linspace(
            start_frame_idx,
            end_frame_idx,
            num=min(
                max_sampled_frames_per_segment,
                end_frame_idx - start_frame_idx + 1,
            ),
            dtype=int,
        )

        previous_gray_frame = None
        frame_differences = []

        for frame_idx in sampled_frame_indices:

            capture.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_idx),
            )

            success, frame = capture.read()

            if not success or frame is None:
                continue

            gray_frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2GRAY,
            )

            gray_frame = cv2.resize(
                gray_frame,
                (160, 90),
            )

            if previous_gray_frame is not None:

                frame_difference = cv2.absdiff(
                    gray_frame,
                    previous_gray_frame,
                )

                normalized_difference = (
                    frame_difference.mean() / 255.0
                )

                frame_differences.append(
                    normalized_difference
                )

            previous_gray_frame = gray_frame

        if frame_differences:

            motion_scores[segment_index] = round(
                float(np.mean(frame_differences)),
                6,
            )

        else:

            motion_scores[segment_index] = 0.0

    capture.release()

    return motion_scores


# ------------------------------------------------------------
# Select Videos for Evidence Generation
# ------------------------------------------------------------

videos_to_process_df = video_inventory_df.copy()

if MAX_VIDEOS_TO_PROCESS != "ALL":

    videos_to_process_df = (
        videos_to_process_df
        .head(MAX_VIDEOS_TO_PROCESS)
        .reset_index(drop=True)
    )

print("Generating evidence metadata records...")
print(f"Videos selected for processing: {len(videos_to_process_df):,}")

# ------------------------------------------------------------
# Generate Fixed-Duration Evidence Records
# ------------------------------------------------------------

evidence_records = []
failed_video_records = []

for video_number, (_, video_row) in enumerate(
    videos_to_process_df.iterrows(),
    start=1,
):

    video_id = str(video_row["video_id"])
    source_video_path = Path(video_row["video_path"])

    capture = cv2.VideoCapture(str(source_video_path))

    if not capture.isOpened():

        failed_video_records.append(
            {
                "video_id": video_id,
                "source_video_path": str(source_video_path),
                "error": "Unable to open video file",
            }
        )

        continue

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    capture.release()

    if fps <= 0 or frame_count <= 0:

        failed_video_records.append(
            {
                "video_id": video_id,
                "source_video_path": str(source_video_path),
                "error": "Invalid FPS or frame count",
            }
        )

        continue

    video_duration_sec = frame_count / fps

    segment_boundaries = []
    current_start_sec = 0.0

    while current_start_sec < video_duration_sec:

        current_end_sec = min(
            current_start_sec + DEFAULT_SEGMENT_DURATION_SEC,
            video_duration_sec,
        )

        current_duration_sec = (
            current_end_sec - current_start_sec
        )

        if (
            current_duration_sec < MIN_SEGMENT_DURATION_SEC
            and segment_boundaries
        ):

            previous_start_sec, _ = segment_boundaries[-1]

            segment_boundaries[-1] = (
                previous_start_sec,
                current_end_sec,
            )

        else:

            segment_boundaries.append(
                (
                    current_start_sec,
                    current_end_sec,
                )
            )

        current_start_sec = current_end_sec

    # --------------------------------------------------------
    # Build Segment Metadata Before Motion Scoring
    # --------------------------------------------------------

    segment_metadata_records = []
    segment_frame_boundaries = []

    for segment_index, (
        start_time_sec,
        end_time_sec,
    ) in enumerate(
        segment_boundaries,
        start=1,
    ):

        midpoint_time_sec = (
            start_time_sec + end_time_sec
        ) / 2.0

        segment_duration_sec = (
            end_time_sec - start_time_sec
        )

        start_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(start_time_sec * fps)),
            ),
        )

        end_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(end_time_sec * fps)) - 1,
            ),
        )

        midpoint_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(midpoint_time_sec * fps)),
            ),
        )

        representative_frame_index = midpoint_frame_idx

        segment_metadata_records.append(
            {
                "segment_index": segment_index,
                "start_time_sec": start_time_sec,
                "midpoint_time_sec": midpoint_time_sec,
                "end_time_sec": end_time_sec,
                "segment_duration_sec": segment_duration_sec,
                "start_frame_idx": start_frame_idx,
                "midpoint_frame_idx": midpoint_frame_idx,
                "end_frame_idx": end_frame_idx,
                "representative_frame_index": representative_frame_index,
            }
        )

        segment_frame_boundaries.append(
            (
                segment_index,
                start_frame_idx,
                end_frame_idx,
            )
        )

    # --------------------------------------------------------
    # Compute Motion Scores Once per Video
    # --------------------------------------------------------

    if COMPUTE_MOTION_SCORE:

        video_motion_scores = compute_video_segment_motion_scores(
            video_path=source_video_path,
            segment_frame_boundaries=segment_frame_boundaries,
        )

    else:

        video_motion_scores = {
            segment_index: 0.0
            for (
                segment_index,
                _,
                _,
            ) in segment_frame_boundaries
        }

    # --------------------------------------------------------
    # Create Evidence Records
    # --------------------------------------------------------

    for segment_record in segment_metadata_records:

        segment_index = segment_record["segment_index"]
        evidence_id = f"{video_id}__ev_{segment_index:04d}"

        evidence_records.append(
            {
                "evidence_id": evidence_id,
                "video_id": video_id,
                "split": video_split_lookup.get(video_id, ""),
                "video_path": str(source_video_path),
                "segment_index": segment_index,
                "evidence_level": EVIDENCE_LEVEL,
                "parent_evidence_id": "",
                "segment_strategy": SEGMENT_STRATEGY,

                "start_time_sec": round(
                    segment_record["start_time_sec"],
                    3,
                ),
                "midpoint_time_sec": round(
                    segment_record["midpoint_time_sec"],
                    3,
                ),
                "end_time_sec": round(
                    segment_record["end_time_sec"],
                    3,
                ),
                "segment_duration_sec": round(
                    segment_record["segment_duration_sec"],
                    3,
                ),

                "start_frame_idx": segment_record[
                    "start_frame_idx"
                ],
                "midpoint_frame_idx": segment_record[
                    "midpoint_frame_idx"
                ],
                "end_frame_idx": segment_record[
                    "end_frame_idx"
                ],
                "representative_frame_index": segment_record[
                    "representative_frame_index"
                ],

                "fps": round(fps, 3),
                "frame_count": frame_count,
                "width": width,
                "height": height,

                "motion_score": video_motion_scores.get(
                    segment_index,
                    0.0,
                ),
                "scene_change_score": DEFAULT_SCENE_CHANGE_SCORE,

                "created_by_notebook": "01_Prepare_Video_Evidence",
            }
        )

    if VERBOSE and video_number % 500 == 0:

        print(
            f"  Processed {video_number:,} / "
            f"{len(videos_to_process_df):,} videos"
        )

# ------------------------------------------------------------
# Assemble Evidence Metadata DataFrame
# ------------------------------------------------------------

evidence_metadata_df = pd.DataFrame.from_records(
    evidence_records,
    columns=EVIDENCE_COLUMNS,
)

failed_videos_df = pd.DataFrame.from_records(
    failed_video_records,
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nEvidence metadata generation complete.")
print(f"Evidence records generated : {len(evidence_metadata_df):,}")
print(f"Videos processed           : {len(videos_to_process_df):,}")
print(f"Failed videos              : {len(failed_videos_df):,}")

if VERBOSE:

    print("\nEvidence Metadata Sample")
    print("-" * 60)
    display(evidence_metadata_df.head())

    if not failed_videos_df.empty:

        print("\nFailed Video Records")
        print("-" * 60)
        display(failed_videos_df)



### 🔷 Step 10 — Validate Evidence Metadata

* Verify that generated evidence records conform to the defined metadata schema.
* Validate required fields, data types, timestamps, and evidence relationships.
* Confirm that evidence records reference valid source videos.
* Identify missing, duplicate, or inconsistent metadata entries.
* Generate validation statistics and quality metrics for the evidence dataset.



In [ ]:
# ============================================================
# Step 10: Validate Evidence Metadata
# ============================================================

# ------------------------------------------------------------
# Run Validation
# ------------------------------------------------------------

validation_summary = validate_evidence_metadata(
    evidence_metadata=evidence_metadata_df,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Convert Validation Issues to DataFrame
# ------------------------------------------------------------

validation_issues_df = (
    validation_issues_to_dataframe(
        validation_summary
    )
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nEvidence metadata validation complete.")

print(
    f"Evidence records validated : "
    f"{validation_summary['record_count']:,}"
)

print(
    f"Errors                     : "
    f"{validation_summary['error_count']}"
)

print(
    f"Warnings                   : "
    f"{validation_summary['warning_count']}"
)

print(
    f"Validation Passed          : "
    f"{validation_summary['passed']}"
)

if VERBOSE and not validation_issues_df.empty:

    print("\nValidation Issues")
    print("-" * 60)

    display(validation_issues_df)



### 🔷 Step 11 — Save Evidence Metadata and Summary Files

* Save the validated evidence metadata dataset to the project output directories.
* Generate summary files describing evidence records and video coverage.
* Export metadata files required for downstream processing and analysis.
* Preserve evidence schema information and processing statistics.
* Verify that all output files were successfully written.



In [ ]:
# ============================================================
# Step 11: Save Evidence Metadata and Summary Files
# ============================================================

# ------------------------------------------------------------
# Build Evidence Summary
# ------------------------------------------------------------

unique_video_count = (
    evidence_metadata_df["video_id"]
    .nunique()
)

average_evidence_per_video = (
    len(evidence_metadata_df)
    / unique_video_count
)

average_segment_duration_sec = (
    evidence_metadata_df["segment_duration_sec"].mean()
)

evidence_summary_records = [
    {
        "metric": "evidence_record_count",
        "value": len(evidence_metadata_df),
    },
    {
        "metric": "unique_video_count",
        "value": unique_video_count,
    },
    {
        "metric": "average_evidence_per_video",
        "value": round(
            average_evidence_per_video,
            2,
        ),
    },
    {
        "metric": "average_segment_duration_sec",
        "value": round(
            average_segment_duration_sec,
            3,
        ),
    },
    {
        "metric": "segment_strategy",
        "value": SEGMENT_STRATEGY,
    },
    {
        "metric": "default_segment_duration_sec",
        "value": DEFAULT_SEGMENT_DURATION_SEC,
    },
    {
        "metric": "validation_passed",
        "value": validation_summary["passed"],
    },
    {
        "metric": "validation_error_count",
        "value": validation_summary["error_count"],
    },
    {
        "metric": "validation_warning_count",
        "value": validation_summary["warning_count"],
    },
]

evidence_summary_df = pd.DataFrame.from_records(
    evidence_summary_records
)

# ------------------------------------------------------------
# Save Evidence Metadata and Summary Files
# ------------------------------------------------------------

evidence_metadata_df.to_csv(
    EVIDENCE_METADATA_CSV,
    index=False,
)

evidence_summary_df.to_csv(
    EVIDENCE_SUMMARY_CSV,
    index=False,
)

# ------------------------------------------------------------
# Save Validation Issues When Present
# ------------------------------------------------------------

EVIDENCE_VALIDATION_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_validation.csv"
)

if not validation_issues_df.empty:

    validation_issues_df.to_csv(
        EVIDENCE_VALIDATION_CSV,
        index=False,
    )

else:

    EVIDENCE_VALIDATION_CSV = None

# ------------------------------------------------------------
# Verify Output Files
# ------------------------------------------------------------

required_output_files = [
    EVIDENCE_METADATA_CSV,
    EVIDENCE_SUMMARY_CSV,
]

for output_file in required_output_files:

    if not output_file.exists():

        raise FileNotFoundError(
            f"Expected output file was not created: "
            f"{output_file}"
        )

# ------------------------------------------------------------
# Display Save Summary
# ------------------------------------------------------------

print(
    "Evidence metadata and summary files "
    "saved successfully."
)

print(
    f"Evidence metadata : "
    f"{EVIDENCE_METADATA_CSV}"
)

print(
    f"Evidence summary  : "
    f"{EVIDENCE_SUMMARY_CSV}"
)

if EVIDENCE_VALIDATION_CSV is not None:

    print(
        f"Validation issues : "
        f"{EVIDENCE_VALIDATION_CSV}"
    )

if VERBOSE:

    print("\nEvidence Summary")
    print("-" * 60)

    display(evidence_summary_df)

    print("\nSaved File Sizes")
    print("-" * 60)

    for output_file in required_output_files:

        file_size_mb = (
            output_file.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{output_file.name:<28} "
            f"{file_size_mb:>10.2f} MB"
        )



### 🔷 Step 12 — Preview Sample Evidence Units

* Display a representative sample of generated evidence metadata records.
* Review evidence identifiers, video references, timestamps, and segment relationships.
* Verify that evidence records accurately represent the intended video segments.
* Inspect summary statistics for the generated evidence dataset.
* Confirm that the evidence metadata is complete and ready for downstream processing.



In [ ]:
# ============================================================
# Step 12: Preview Sample Evidence Units
# ============================================================

# ------------------------------------------------------------
# Select Sample Evidence Records
# ------------------------------------------------------------

sample_evidence_df = (
    evidence_metadata_df
    .sample(
        n=min(SAMPLE_VIDEO_COUNT, len(evidence_metadata_df)),
        random_state=42,
    )
    .sort_values(
        by=[
            "video_id",
            "segment_index",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display Sample Evidence Records
# ------------------------------------------------------------

print("Sample evidence units selected.")
print(f"Sample evidence records : {len(sample_evidence_df)}")

if VERBOSE:

    print("\nSample Evidence Metadata")
    print("-" * 60)

    display(sample_evidence_df)

# ------------------------------------------------------------
# Display Evidence Coverage by Split
# ------------------------------------------------------------

evidence_split_summary_df = (
    evidence_metadata_df
    .groupby("split", dropna=False)
    .agg(
        evidence_count=("evidence_id", "count"),
        unique_video_count=("video_id", "nunique"),
        average_segment_duration_sec=(
            "segment_duration_sec",
            "mean",
        ),
    )
    .reset_index()
)

evidence_split_summary_df["average_segment_duration_sec"] = (
    evidence_split_summary_df["average_segment_duration_sec"]
    .round(3)
)

print("\nEvidence coverage by split:")

display(evidence_split_summary_df)

# ------------------------------------------------------------
# Display Evidence Duration Summary
# ------------------------------------------------------------

evidence_duration_summary_df = (
    evidence_metadata_df["segment_duration_sec"]
    .describe()
    .to_frame(name="segment_duration_sec")
)

print("\nEvidence segment duration summary:")

display(evidence_duration_summary_df)



### 🔷 Step 13 — Notebook Summary

* Review the evidence metadata generation process and resulting outputs.
* Summarize video coverage, evidence record counts, and validation results.
* Confirm that the evidence metadata dataset was successfully generated and saved.
* Verify readiness for downstream processing and analysis.
* Identify any issues or recommendations for subsequent notebooks.



In [ ]:
# ============================================================
# Step 13: Notebook Summary
# ============================================================

# ------------------------------------------------------------
# Summarize Notebook Outputs
# ------------------------------------------------------------

print("Notebook 02 complete.")
print("=" * 60)

print("\nPrimary Output")
print("-" * 60)
print(f"Evidence metadata CSV : {EVIDENCE_METADATA_CSV}")
print(f"Evidence summary CSV  : {EVIDENCE_SUMMARY_CSV}")

print("\nEvidence Generation Summary")
print("-" * 60)
print(f"Videos processed          : {evidence_metadata_df['video_id'].nunique():,}")
print(f"Evidence records created  : {len(evidence_metadata_df):,}")
print(f"Segmentation strategy     : {SEGMENT_STRATEGY}")
print(f"Default segment duration  : {DEFAULT_SEGMENT_DURATION_SEC:.1f} seconds")

print("\nValidation Summary")
print("-" * 60)
print(f"Validation passed         : {validation_summary['passed']}")
print(f"Validation errors         : {validation_summary['error_count']}")
print(f"Validation warnings       : {validation_summary['warning_count']}")

